In [1]:
from mdagent import MDAgent

In [2]:
import nest_asyncio
nest_asyncio.apply()

In [3]:
mda = MDAgent(agent_type='Structured', model='gpt-4-1106-preview', use_memory=False, top_k_tools="all")

In [4]:
#llm='gpt-4-1106-preview'
prompt = '''Download and clean protein 1A3N'''

In [5]:
answer = mda.run(prompt)

Thought: The task is to download and clean the protein with PDB ID 1A3N. I will use the PDBFileDownloader tool to download the protein file and then use the CleaningToolFunction to clean the downloaded file.

Action:
```
{
  "action": "PDBFileDownloader",
  "action_input": {
    "query": "1A3N"
  }
}
```PDB file found with this ID: 1A3N


2024-06-29 23:44:56.204 
  command:

    streamlit run /Users/samcox/anaconda3/envs/mda_jun29/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]


Now that the PDB file for protein 1A3N has been downloaded, the next step is to clean the file. I will use the CleaningToolFunction to remove heterogens, add missing atoms and hydrogens, replace nonstandard residues, and remove water. This will provide a cleaned version of the protein structure.

Action:
```
{
  "action": "CleaningToolFunction",
  "action_input": {
    "pdb_id": "1A3N_234456",
    "replace_nonstandard_residues": true,
    "add_missing_atoms": true,
    "remove_heterogens": true,
    "remove_water": true,
    "add_hydrogens": true,
    "add_hydrogens_ph": 7.0
  }
}
```file path: /Users/samcox/Desktop/md-agent/ckpt/ckpt_146/pdb/1A3N_raw_20240629_234456.pdb
Final Answer: The protein with PDB ID 1A3N has been successfully downloaded and cleaned. The cleaned file is available with the ID 1A3N_234507.

### make sure files actually exist

In [7]:
registry = mda.path_registry

In [8]:
all_names = registry.list_path_names()

pdb_raw_id = "1A3N_234456"
pdb_cleaned = "1A3N_234507"

In [9]:
all_names

'Names found in registry: 1A3N_234456, 1A3N_234507'

In [13]:
import os
assert os.path.exists(registry.get_mapped_path(pdb_raw_id))
assert os.path.exists(registry.get_mapped_path(pdb_cleaned))

In [15]:
file1 = open(registry.get_mapped_path(pdb_raw_id), "r")
file1.read()

file2 = open(registry.get_mapped_path(pdb_cleaned), "r")
file2.read()

assert file1 != file2

In [16]:
import mdtraj as md

traj1 = md.load(registry.get_mapped_path(pdb_raw_id))
traj2 = md.load(registry.get_mapped_path(pdb_cleaned))

In [37]:
atoms1 = [atom.name for atom in traj1.topology.atoms]
atoms2 = [atom.name for atom in traj2.topology.atoms]

atoms2_only = set(atoms2).difference(set(atoms1))

#assert that atoms2_only only contains hydrogen atoms
assert all([atom.startswith('H') for atom in atoms2_only])